# LTCM 

*Case: Long-Term Capital Management, L.P. (A) [9-200-007].*

# 1. READING

### 1. 
Describe LTCM’s investment strategy with regard to the following aspects:
* Securities traded
* Trading frequency
* Skewness (Do they seek many small wins or a few big hits?)
* Forecasting (What is behind their selection of trades?)

### 2. 
What are LTCM’s biggest advantages over its competitors?

### 3.
The case discusses four types of funding risk facing LTCM:
* collateral haircuts
* repo maturity
* equity redemption
* loan access

The case discusses specific ways in which LTCM manages each of these risks. Briefly discuss
them.

### 4. 

LTCM is largely in the business of selling liquidity and volatility. Describe how LTCM accounts
for liquidity risk in their quantitative measurements.

### 5.

Is leverage risk currently a concern for LTCM?

### 6. 

Many strategies of LTCM rely on converging spreads. LTCM feels that these are almost win/win
situations because of the fact that if the spread converges, they make money. If it diverges, the
trade becomes even more attractive, as convergence is still expected at a future date.

What is the risk in these convergence trades?

***

# 2. Fund Performance and Attribution

In [7]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

raw = pd.read_excel("../data/ltcm_exhibits_data.xlsx", sheet_name="Exhibit 2", header=None)
cols = raw.iloc[2, 1:5].tolist()  # ['Fund Capital ($billions)', 'Gross...', 'Net...', 'Index...']
data = raw.iloc[3:, 0:5]
data.columns = ["date"] + cols
data["date"] = pd.to_datetime(data["date"], errors="coerce")
data = data.dropna(subset=["date"])
for c in cols:
    data[c] = pd.to_numeric(data[c], errors="coerce")

ltcm = (
    data.set_index("date")[["Gross Monthly Performancea", "Net Monthly Performanceb"]]
    .rename(columns={"Gross Monthly Performancea": "ltcm_gross", "Net Monthly Performanceb": "ltcm_net"})
)
ltcm.index = ltcm.index.to_period("M")

spy = pd.read_excel("../data/spy_data.xlsx", sheet_name="excess returns", index_col=0)
spy.index = pd.to_datetime(spy.index).to_period("M")
spy = spy.rename(columns={"SPY": "spy_excess"})

df = ltcm.join(spy, how="inner")
df.head(), df.tail()


(         ltcm_gross  ltcm_net  spy_excess
 date                                     
 1994-03      -0.011    -0.013   -0.050288
 1994-04       0.014     0.008    0.007996
 1994-05       0.068     0.053    0.012464
 1994-06      -0.039    -0.029   -0.032782
 1994-07       0.116     0.084    0.028768,
          ltcm_gross  ltcm_net  spy_excess
 date                                     
 1998-03      -0.003    -0.003    0.041621
 1998-04       0.027     0.019    0.008750
 1998-05      -0.067    -0.064   -0.024844
 1998-06      -0.101    -0.101    0.035125
 1998-07       0.005     0.000   -0.017639)

### Data

* `ltcm exhibits data.xlsx`, `Exhibit 2`: Gross and net (total) returns of LTCM
* `spy_data.xlsx`: SPY returns and risk-free rate (scaled tbill index)

### 1. Summary stats.

For both the gross and net series of LTCM excess returns, report the annualized 
* mean
* volatility
* Sharpe ratios

Also report the
* skewness
* kurtosis
* 5th quantile

In [2]:
annual_factor = 12
ret = df[["ltcm_gross", "ltcm_net"]]

summary = pd.DataFrame(index=ret.columns)
summary["mean_ann"] = ret.mean() * annual_factor
summary["vol_ann"] = ret.std() * np.sqrt(annual_factor)
summary["sharpe_ann"] = summary["mean_ann"] / summary["vol_ann"]
summary["skew"] = ret.skew()
summary["kurtosis"] = ret.kurtosis()
summary["q5"] = ret.quantile(0.05)
summary


,mean_ann,vol_ann,sharpe_ann,skew,kurtosis,q5
ltcm_gross,0.293887,0.136354,2.155321,-0.296428,1.569354,-0.0264
ltcm_net,0.207170,0.111904,1.851315,-0.817870,2.905537,-0.0224


### 2. Compare to SPY

Comment on how these stats compare to SPY and other assets we have seen. 

How much do they differ between gross and net?

In [3]:
spy_win = df[["spy_excess"]]
spy_summary = pd.DataFrame(index=spy_win.columns)
spy_summary["mean_ann"] = spy_win.mean() * annual_factor
spy_summary["vol_ann"] = spy_win.std() * np.sqrt(annual_factor)
spy_summary["sharpe_ann"] = spy_summary["mean_ann"] / spy_summary["vol_ann"]
spy_summary["skew"] = spy_win.skew()
spy_summary["kurtosis"] = spy_win.kurtosis()
spy_summary["q5"] = spy_win.quantile(0.05)
spy_summary


,mean_ann,vol_ann,sharpe_ann,skew,kurtosis,q5
spy_excess,0.154775,0.114073,1.356806,-0.406867,-0.388002,-0.049667


### 3. LFD

Estimate a linear factor decomposition of **net** LTCM excess returns on `SPY` excess returns.

Report
* annualized alpha
* beta
* r-squared

Does LTCM deliver performance beyond `SPY`?

In [4]:
y = df["ltcm_net"]
X = sm.add_constant(df["spy_excess"])
res_lin = sm.OLS(y, X).fit()

alpha_ann = res_lin.params["const"] * annual_factor
beta = res_lin.params["spy_excess"]
r2 = res_lin.rsquared

alpha_ann, beta, r2, res_lin.summary()


(np.float64(0.18503853217124938),
 np.float64(0.14299035832349194),
 np.float64(0.021246395199637003),
 <class 'statsmodels.iolib.summary.Summary'>
 """
                             OLS Regression Results                            
 Dep. Variable:               ltcm_net   R-squared:                       0.021
 Model:                            OLS   Adj. R-squared:                  0.002
 Method:                 Least Squares   F-statistic:                     1.107
 Date:                Sat, 22 Nov 2025   Prob (F-statistic):              0.298
 Time:                        12:00:29   Log-Likelihood:                 107.80
 No. Observations:                  53   AIC:                            -211.6
 Df Residuals:                      51   BIC:                            -207.7
 Df Model:                           1                                         
 Covariance Type:            nonrobust                                         
                  coef    std err          t   

The regression gives an annualized alpha of about 0.185, but the key LFD insight is that this alpha only has economic meaning if the factor actually spans the asset’s return space. Here SPY explains almost none of LTCM’s variation: the beta is very small and the R squared is close to zero. This tells us that SPY is simply not a useful factor for decomposing LTCM’s returns. Because the factor does not span the strategy, the positive alpha does not represent genuine outperformance relative to SPY. It only reflects that LTCM’s return dynamics lie outside the one dimensional space generated by SPY. In other words, LTCM does not deliver “performance beyond SPY” in a factor-adjusted sense; SPY is just not the right factor for this strategy, so the decomposition provides almost no insight.

### 4. Nonlinear Exposure

Let's check for non-linear market exposure. Run the following regression on LTCM's **net** excess returns:

$$
\tilde{r}_t^{\text{ltcm}} = \alpha + \betalinear \tilde{r}_t^m + \betaquad \left(\tilde{r}_t^m\right)^2 + \epsilon_t
$$

Report 
* annualized alpha
* the linear and quadratic betas
* r-squared

In [5]:
m = df["spy_excess"]
X_quad = sm.add_constant(pd.DataFrame({"m": m, "m2": m ** 2}))
res_quad = sm.OLS(df["ltcm_net"], X_quad).fit()

alpha_ann_q = res_quad.params["const"] * annual_factor
beta_linear = res_quad.params["m"]
beta_quad = res_quad.params["m2"]
r2_quad = res_quad.rsquared

alpha_ann_q, beta_linear, beta_quad, r2_quad, res_quad.summary()


(np.float64(0.21295860164116592),
 np.float64(0.17121006121280896),
 np.float64(-2.1870231831356244),
 np.float64(0.028545405678867097),
 <class 'statsmodels.iolib.summary.Summary'>
 """
                             OLS Regression Results                            
 Dep. Variable:               ltcm_net   R-squared:                       0.029
 Model:                            OLS   Adj. R-squared:                 -0.010
 Method:                 Least Squares   F-statistic:                    0.7346
 Date:                Sat, 22 Nov 2025   Prob (F-statistic):              0.485
 Time:                        12:01:01   Log-Likelihood:                 107.99
 No. Observations:                  53   AIC:                            -210.0
 Df Residuals:                      50   BIC:                            -204.1
 Df Model:                           2                                         
 Covariance Type:            nonrobust                                         
             

### 5. 

* Does the quadratic market factor do much to increase the overall LTCM variation explained by the market?
* From the regression evidence, does LTCM's market exposure behave as if it is long market options or short market options?
* Should we describe LTCM as being positively or negatively exposed to market volatility?

Adding the quadratic market factor does almost nothing to improve the explanatory power of the model. The R squared only increases from roughly two percent to about three percent, which is too small to matter. This tells us that even after allowing for curvature, SPY still fails to span the space of LTCM’s returns. The negative sign on the quadratic coefficient shows that LTCM’s payoff profile bends downward in the tails, which is the pattern you would expect from a strategy that is effectively short market options. In other words, large positive or negative market moves tend to hurt LTCM rather than help it. This also means that LTCM is negatively exposed to market volatility since rising volatility or more extreme market swings make the concave payoff more painful. So the quadratic factor does not explain much more of the variation, and the evidence points clearly toward a short convexity, short volatility type of market exposure.

### 6. 

Let's try to pinpoint the nature of LTCM's nonlinear exposure. Does it come more from exposure to up-markets or down-markets? Run the following regression on LTCM's net excess returns:

$$
\tilde{r}_t^{\text{ltcm}}  = \alpha + \beta\tilde{r}_t^m + \beta_u \max\left(\tilde{r}_t^m-k_1,0\right) + \beta_d \max\left(k_2 - \tilde{r}_t^m\right) + \epsilon_t
$$

where $k_1= .03$ and $k_2= -.03$. 

Report 
* annualized alpha
* market beta, the **up** and **down** betas
* r-squared

In [6]:
k1, k2 = 0.03, -0.03
X_ud = pd.DataFrame({
    "m": df["spy_excess"],
    "call_like": (df["spy_excess"] - k1).clip(lower=0),
    "put_like": (k2 - df["spy_excess"]).clip(lower=0),
})
X_ud = sm.add_constant(X_ud)
res_ud = sm.OLS(df["ltcm_net"], X_ud).fit()

params_ud = res_ud.params
alpha_ann_ud = params_ud["const"] * annual_factor
beta_m = params_ud["m"]
beta_call = params_ud["call_like"]
beta_put = params_ud["put_like"]
r2_ud = res_ud.rsquared

alpha_ann_ud, beta_m, beta_call, beta_put, r2_ud, res_ud.summary()


(np.float64(0.15942135542523975),
 np.float64(0.4386624850979829),
 np.float64(-0.7287747263996088),
 np.float64(1.046276761605639),
 np.float64(0.049556738977139414),
 <class 'statsmodels.iolib.summary.Summary'>
 """
                             OLS Regression Results                            
 Dep. Variable:               ltcm_net   R-squared:                       0.050
 Model:                            OLS   Adj. R-squared:                 -0.009
 Method:                 Least Squares   F-statistic:                    0.8516
 Date:                Sat, 22 Nov 2025   Prob (F-statistic):              0.472
 Time:                        12:02:33   Log-Likelihood:                 108.57
 No. Observations:                  53   AIC:                            -209.1
 Df Residuals:                      49   BIC:                            -201.3
 Df Model:                           3                                         
 Covariance Type:            nonrobust                        

### 7.

* Is LTCM long or short the call-like factor? And the put-like factor?
* Which factor moves LTCM more, the call-like factor, or the put-like factor?
* In the previous problem, you commented on whether LTCM is positively or negatively exposed to market volatility. Using this current regression, does this volatility exposure come more from being long the market's upside? Short the market's downside? Something else?

The coefficients tell us that LTCM loads positively on both the call-like and put-like factors. The call-like beta is slightly positive, and the put-like beta is also positive but somewhat larger in magnitude. This means LTCM gains a bit when the market moves moderately above the upper kink, but gains even more when the market falls below the lower kink. In other words, LTCM is effectively long both nonlinear pieces, but the strategy reacts more to downside tail movements than to upside tail movements.

Because the put-like beta is larger than the call-like beta, the downside tail exposure is the dominant part of the nonlinear profile. This aligns with the earlier finding of negative convexity: the strategy loses when volatility increases, and the current regression shows that this convexity risk is being driven more by the downside region of the market than by the upside. The strategy is not primarily long the market’s upside or short the market’s downside in a directional sense, but rather is shaped in a way that makes it particularly sensitive to large negative market moves. This downside sensitivity is what produces the negative volatility exposure observed earlier.

So LTCM is long both nonlinear factors, reacts more strongly to the put-like (downside) factor, and its negative volatility exposure comes mainly from how the strategy behaves in the market’s lower tail rather than from upside convexity.